# Lab 4: Motors and Open Loop Control
Motor driver testing, calibration, and open-loop driving.

In [58]:
!pip install bleak colorama numpy matplotlib bleach PyYAML  --upgrade


In [59]:
%load_ext autoreload
%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np
import asyncio
import matplotlib.pyplot as plt

LOG.propagate = False

PWM_MAX = (1 << 16) - 1  # 65535, matches Arduino 16-bit resolution


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [60]:
ble = get_ble_controller()
ble.connect()


2026-03-03 16:11:45,685 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:81:95:21:a2:64
2026-03-03 16:11:45,685 | INFO     |: Scanning for device with address: c0:81:95:21:a2:64, service UUID: 1785129f-3b3a-4cf5-a01f-03668e8b12e9
2026-03-03 16:11:55,735 | INFO     |: Found 0 device(s) advertising service 1785129f-3b3a-4cf5-a01f-03668e8b12e9


Exception: Could not find device advertising service uuid: 1785129f-3b3a-4cf5-a01f-03668e8b12e9

## Notification Handler Setup
Handles motor feedback (`CAL:`), sensor data, and `END` markers.

In [ ]:
transfer_done = False
last_message = ""

def handle_data(uuid, message):
    global transfer_done, last_message
    msg = message.decode()
    last_message = msg
    print(msg)
    if msg == "END":
        transfer_done = True

try:
    ble.stop_notify(ble.uuid["RX_STRING"])
except Exception:
    pass

ble.start_notify(ble.uuid["RX_STRING"], handle_data)
print("Notification handler ready")


Notification handler ready


---
## 1. Safety Timeout
Always set a timeout before driving motors. Motors auto-stop after this many milliseconds.

In [ ]:
ble.send_command(CMD.MOTOR_TIMEOUT, "3000")  # 3 second safety shutoff


---
## 2. Single Motor Testing (Wheels Elevated)
Test each motor individually in both directions. Use external power supply first, then battery.

**Pin mapping:** Motor 1 (left) = A0/A1, Motor 2 (right) = A2/A3

In [ ]:
# Motor 1 forward
ble.send_command(CMD.MOTOR_CMD, "30000|0")
time.sleep(2)
ble.send_command(CMD.MOTOR_STOP, "")


In [ ]:
# Motor 1 reverse
ble.send_command(CMD.MOTOR_CMD, "-200|0")
time.sleep(2)
ble.send_command(CMD.MOTOR_STOP, "")


### Motor 2

In [ ]:
# Motor 2 forward
ble.send_command(CMD.MOTOR_CMD, "0|200")
time.sleep(2)
ble.send_command(CMD.MOTOR_STOP, "")


In [ ]:
# Motor 2 reverse
ble.send_command(CMD.MOTOR_CMD, "0|-200")
time.sleep(2)
ble.send_command(CMD.MOTOR_STOP, "")


### Both Motors Together

In [ ]:
# Both motors forward
ble.send_command(CMD.MOTOR_CMD, "200|200")
time.sleep(2)
ble.send_command(CMD.MOTOR_STOP, "")


In [ ]:
# Both motors reverse
ble.send_command(CMD.MOTOR_CMD, "-200|-200")
time.sleep(2)
ble.send_command(CMD.MOTOR_STOP, "")


---
## 3. Minimum PWM Threshold
Sweep PWM values to find the lowest value that moves the robot on the ground.
Place the robot on the floor before running.

In [ ]:
# Sweep from low to high -- watch which value first moves the robot
for pwm in range(5000, 40000, 2500):
    print(f"Testing PWM = {pwm}")
    ble.send_command(CMD.MOTOR_CMD, f"{pwm}|{pwm}")
    time.sleep(1.5)
    ble.send_command(CMD.MOTOR_STOP, "")
    time.sleep(1.0)

print("Sweep done")


Testing PWM = 5000
Testing PWM = 7500
Testing PWM = 10000
Testing PWM = 12500
Testing PWM = 15000
Testing PWM = 17500
Testing PWM = 20000
Testing PWM = 22500
Testing PWM = 25000
Testing PWM = 27500
Testing PWM = 30000
Testing PWM = 32500
Testing PWM = 35000
Testing PWM = 37500


BleakError: disconnected

---
## 4. Motor Calibration
If the robot veers left/right, adjust the calibration factor for motor 2.
- `> 1.0` if motor 2 is slower (robot veers right)
- `< 1.0` if motor 2 is faster (robot veers left)

In [ ]:
ble.send_command(CMD.MOTOR_CAL, "1.0")  # adjust as needed

# Test straight line after calibration
ble.send_command(CMD.MOTOR_CMD, "30000|30000")
time.sleep(3)
ble.send_command(CMD.MOTOR_STOP, "")


CAL:1.000


---
## 5. Straight Line Drive (2+ meters)
Drive forward in a straight line for at least 2 meters. Record video for write-up.

In [ ]:
# Adjust PWM and duration for your robot + floor surface
DRIVE_PWM = 30000
DRIVE_TIME = 4  # seconds

ble.send_command(CMD.MOTOR_TIMEOUT, "5000")
ble.send_command(CMD.MOTOR_CMD, f"{DRIVE_PWM}|{DRIVE_PWM}")
time.sleep(DRIVE_TIME)
ble.send_command(CMD.MOTOR_STOP, "")


---
## 6. Open Loop with Turns
Demonstrate untethered open-loop control: forward, turn, forward, etc.

In [ ]:
ble.send_command(CMD.MOTOR_TIMEOUT, "10000")

# Forward
ble.send_command(CMD.MOTOR_CMD, "30000|30000")
time.sleep(2)

# Spin turn (opposite wheels)
ble.send_command(CMD.MOTOR_CMD, "30000|-30000")
time.sleep(0.5)

# Forward again
ble.send_command(CMD.MOTOR_CMD, "30000|30000")
time.sleep(2)

# Turn the other way
ble.send_command(CMD.MOTOR_CMD, "-30000|30000")
time.sleep(0.5)

# Forward
ble.send_command(CMD.MOTOR_CMD, "30000|30000")
time.sleep(2)

ble.send_command(CMD.MOTOR_STOP, "")


---
## Disconnect

In [ ]:
ble.send_command(CMD.MOTOR_STOP, "")
# ble.disconnect()
